# Notebook 7: Dynamic Multi-Regime Audio & Oscilloscope Instrument

This notebook demonstrates the **Phase 2.5 (`v1.4.5-rc1`)** multi-regime hardware architecture on the PYNQ-Z2.

### 🌟 Multi-Regime Capabilities:
- 🔄 **Runtime Configurable Decimation ($M=1, 10, 20, 50$):** Switch between High-Speed Lab Oscilloscope ($0 - 250\,\text{kHz}$) and Deep Bass Zoom ($0 - 5\,\text{kHz}$) on the fly.
- 🪟 **Multi-Windowing Engine:** Dynamic selection of `Hann`, `Blackman`, `Flat-Top`, and `Rectangular` windows for $>58\,\text{dB}$ sidelobe suppression.
- 🔊 **Jupyter Audio Playback:** Listen to real-time recorded microphone audio directly in the notebook.
- 🎛 **Interactive Multi-Regime Dashboard:** Complete dual-channel live instrument with automatic timebase adaptation.

## 1. System Setup & Overlay Initialization

In [ ]:
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay
from pynq_oscilloscope.fft_dma import StreamingFFT
import numpy as np
import matplotlib.pyplot as plt

# Ensure USB permissions
check_usb_permissions()

# Load Multi-Regime Overlay (v1.4.5-rc1)
ol = OscilloscopeOverlay()

print("✅ Multi-Regime Overlay loaded successfully!")
print("Active Configuration:", ol.get_profile_info())

## 2. Dynamic On-the-Fly Profile Switching
Switch between all 4 operating profiles in software without resetting the FPGA or restarting the kernel.

In [ ]:
profiles = ["oscilloscope", "audio", "speech", "bass_zoom"]

print("🔄 Testing Dynamic Profile Switching:")
for p in profiles:
    info = ol.set_profile(p)
    print(f"  • Mode: {info['mode']:<14} | M={info['decimation_M']:<2} | fs={info['sample_rate_hz']/1e3:5.1f} kSPS | Window={info['time_window_ms']:6.2f} ms | Δf={info['delta_f_hz']:5.2f} Hz | Max Freq={info['max_frequency_hz']/1e3:5.1f} kHz")

## 3. Windowing Function Comparison (Spectral Leakage Suppression)
Capture a microphone frame and compare the spectral response under **Hann**, **Blackman** ($>58\,\text{dB}$ rejection), **Flat-Top** (calibrated amplitude), and **Rectangular** windows.

In [ ]:
# Set profile to standard audio
ol.set_profile("audio")
ol.trigger.configure(mode="Auto", threshold_volts=1.65)

# Capture microphone audio
v_a0, v_a1 = ol.capture_stereo()

windows = ["rectangular", "hann", "blackman", "flattop"]
colors = ["#888888", "#00FFCC", "#FF007F", "#FFA500"]

plt.figure(figsize=(11, 5), dpi=100)
for win, col in zip(windows, colors):
    freqs, mags = ol.fft.compute_spectrum(v_a0, unit="dBV", window_type=win)
    plt.plot(freqs, mags, label=f"{win.capitalize()} Window", color=col, linewidth=1.4, alpha=0.85)

plt.title("Window Function Sidelobe Suppression Comparison on Microphone Audio", fontsize=11, fontweight="bold")
plt.xlabel("Frequency (Hz)", fontsize=10)
plt.ylabel("Magnitude (dBV)", fontsize=10)
plt.xlim(0, 5000)
plt.ylim(-100, 0)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 4. Deep Bass Zoom Mode Test ($10\,\text{kSPS}, \Delta f = 4.88\,\text{Hz}, T_{\text{window}} = 204.8\,\text{ms}$)
Switch to `bass_zoom` mode. The time window expands to **$204.8\,\text{ms}$**, capturing multiple full cycles of low-frequency bass notes with high frequency resolution.

In [ ]:
# Activate Deep Bass Zoom Mode (M=50)
info = ol.set_profile("bass_zoom")
print(f"🎸 Deep Bass Zoom Active: Window = {info['time_window_ms']:.1f} ms | Δf = {info['delta_f_hz']:.2f} Hz per bin")

v_a0, v_a1 = ol.capture_stereo()
time_ms = np.linspace(0, info["time_window_ms"], len(v_a0))

# Compute Hann-windowed bass spectrum
freqs_bass, mags_bass = ol.fft.compute_spectrum(v_a0, unit="dBV", window_type="hann")
peak_f, peak_m = StreamingFFT.get_peak_frequency(freqs_bass, mags_bass, min_freq_hz=10.0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), dpi=100)

# Time plot (204.8 ms)
ax1.plot(time_ms, v_a0, color="#00A389", linewidth=1.4, label="Mic 1: A0 (Deep Bass Window)")
ax1.axhline(1.65, color="#FFA500", linestyle="--", alpha=0.7, label="1.65V Bias")
ax1.set_title(f"Deep Bass Capture: 204.8 ms Observation Window @ 10 kSPS", fontsize=11, fontweight="bold")
ax1.set_xlabel("Time (Milliseconds)", fontsize=10)
ax1.set_ylabel("Voltage (V)", fontsize=10)
ax1.set_ylim(0, 3.3)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="upper right")

# Spectrum (0 to 1000 Hz zoom)
ax2.plot(freqs_bass, mags_bass, color="#FF007F", linewidth=1.5, label=f"Sub-Bass Spectrum (Peak: {peak_f:.1f} Hz)")
ax2.scatter([peak_f], [peak_m], color="#00FFCC", s=50, zorder=5, label="Detected Fundamental")
ax2.set_title("High-Resolution Sub-Bass Spectrum (Δf = 4.88 Hz)", fontsize=11, fontweight="bold")
ax2.set_xlabel("Frequency (Hz)", fontsize=10)
ax2.set_ylabel("Magnitude (dBV)", fontsize=10)
ax2.set_xlim(0, 1000)
ax2.set_ylim(-90, 0)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 5. In-Notebook Audio Recording & Playback
Record audio through the FPGA DMA from Microphone 1 and listen to it directly in the browser.

In [ ]:
# Set to Full Audio mode for audio playback
ol.set_profile("audio")

print("🔊 Playing captured Microphone 1 audio:")
ol.play_audio(channel=1)

## 6. Launch the Interactive Multi-Regime Dashboard
Launch the complete **`AudioDashboard`** with the live **Regime Selector** and **Window Dropdown**.

In [ ]:
# Launch Multi-Regime Audio Dashboard
app = ol.audio_dashboard()

## 7. Clean Hardware Shutdown

In [ ]:
app.stop()
ol.close()
print("🔒 Multi-regime instrument stopped and memory released.")